# 2 · Legal-citation extraction: counts, agreement, citation reasons

This notebook produces the results of **Section "Legal citation extraction"** of the paper
and the corresponding appendix tables:

| Output in this notebook | Paper table / claim |
|---|---|
| Average number of legal references per present issue | Table `tab:cit_counts` |
| Pairwise agreement on citation extraction, N=20 (pooled) | Table `tab:cit_pairwise` |
| Pairwise agreement, per-issue averaged | Appendix Table `tab:cit_pairwise_macro` |
| LLM vs each annotator on the full N=35 sets | Tables `tab:cit_n35`, `tab:cit_n35_macro` |
| Union / intersection ground truth (N=50) | Appendix Table `tab:cit_union_inter` |
| Citation-reason faithfulness | Table `tab:cit_reason` and in-text |

**Scope.** All citation metrics refer to the LLM output **after** the hallucination filter,
and are restricted to issues marked as present (an issue marked absent by any annotator is
dropped — the "joint-present" rule). Inter-annotator citation overlap per issue
(|A1 ∩ A2|) comes from `shared_citations.csv`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA = '../data'

# Expert annotations of the LLM extraction on the 50 test judgments
# (one row per LLM-extracted issue). Annotator mapping used in the paper:
#   A1 = Alessia, A2 = Piera.
df_a = pd.read_csv(f'{DATA}/validation_annotator_A1.csv')   # A1
df_p = pd.read_csv(f'{DATA}/validation_annotator_A2.csv')   # A2

sentenze_a = set(df_a['Sentenza'])
sentenze_p = set(df_p['Sentenza'])
common_sentenze = sentenze_a & sentenze_p

print(f"A1: {len(df_a)} extracted issues over {df_a['Sentenza'].nunique()} judgments")
print(f"A2: {len(df_p)} extracted issues over {df_p['Sentenza'].nunique()} judgments")
print(f"Judgments annotated by both (shared set): {len(common_sentenze)}")

A1: 54 extracted issues over 35 judgments
A2: 54 extracted issues over 35 judgments
Judgments annotated by both (shared set): 20


In [2]:
# ── Column-name constants (the CSV headers are in Italian) ──────────────────
COL_PRESENT = 'questione presente nella sentenza (TRUE/FALSE)'
COL_NOT_EXTRACTED = ("# questioni presenti NON estratte (numero minimo di questioni presenti "
                     "che in aggiunta a quelle estratte coprirebbero l'intero contenuto della sentenza)")

COL_CIT_ESPERTO  = "# citazioni esperto (# di elementi nella 'Lista citazioni esperto')"
COL_CIT_ESTRATTE = '# citazioni estratte (contare i riferimenti di diritto in J)'
COL_CIT_IN_ESP   = ("# citazioni estratte presenti nella lista dell'esperto "
                    "(# citazioni estratte dall'LLM che appaiono anche nella lista dell'esperto")

COL_MOTIVI_ESTRATTI = '# motivi citazione estratti'
COL_SCORE_MOTIVI    = ('somma score motivi citazione (per ogni motivo 1 se il motivo corrisponde '
                       'a quanto detto nella sentenza (ovvero se il motivo è contenuto nella '
                       'sentenza e se corrisponde con quanto scritto) e 0 altrimenti)')

In [3]:
# ── Precision / recall / F1 helpers ──────────────────────────────────────────
def prf1(n_x, n_y, n_xy):
    """Precision (of X wrt Y), Recall (of Y found in X), F1.
    F1 is 0 when at least one of P,R is 0 (even if the other is undefined).
    F1 is NaN only when both denominators are 0."""
    p = n_xy / n_x if n_x > 0 else np.nan
    r = n_xy / n_y if n_y > 0 else np.nan
    if np.isnan(p) and np.isnan(r):
        f = np.nan
    elif np.isnan(p) or np.isnan(r):
        f = 0.0   # one side is 0 (n_xy must be 0)
    elif (p + r) == 0:
        f = 0.0
    else:
        f = 2 * p * r / (p + r)
    return p, r, f

def micro_prf1_cols(df, col_x, col_y, col_xy):
    """Pooled metrics: sum counts over all units, then compute P/R/F1."""
    return prf1(df[col_x].sum(), df[col_y].sum(), df[col_xy].sum())

def macro_prf1_cols(df, col_x, col_y, col_xy):
    # Average precision and recall over the units where each is defined, then
    # take F1 as their HARMONIC MEAN, so macro-F1 always lies between macro-P
    # and macro-R. Averaging per-unit F1 independently is unsound here:
    # one-sided units (one party lists 0 items) are forced to F1=0 yet are
    # dropped from the P or R average (their P or R is undefined/NaN), which can
    # push the mean per-unit F1 BELOW both macro-P and macro-R.
    ps, rs = [], []
    for _, row in df.iterrows():
        p, r, _ = prf1(row[col_x], row[col_y], row[col_xy])
        if not np.isnan(p): ps.append(p)
        if not np.isnan(r): rs.append(r)
    P = np.mean(ps) if ps else np.nan
    R = np.mean(rs) if rs else np.nan
    if np.isnan(P) and np.isnan(R):
        F = np.nan
    elif np.isnan(P) or np.isnan(R) or (P + R) == 0:
        F = 0.0
    else:
        F = 2 * P * R / (P + R)
    return P, R, F

fmt_pct = lambda v: f"{v:.1%}" if not np.isnan(v) else "---"


def bootstrap_micro_prf1(df, comparisons, B=10_000, seed=0):
    """95% percentile confidence intervals for the pooled P/R/F1.

    Non-parametric cluster bootstrap at the judgment level: the rows of `df`
    (one row per judgment, holding that judgment's counts) are resampled with
    replacement B times and the pooled metric is recomputed on each replicate.
    Resampling whole judgments preserves the dependence between items
    belonging to the same decision. Replicates where a denominator is 0 are
    skipped for that metric (percentiles ignore NaNs).
    """
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(df), size=(B, len(df)))
    res = {}
    for label, cx, cy, cxy in comparisons:
        X  = df[cx].to_numpy(float)[idx].sum(axis=1)
        Y  = df[cy].to_numpy(float)[idx].sum(axis=1)
        XY = df[cxy].to_numpy(float)[idx].sum(axis=1)
        with np.errstate(divide='ignore', invalid='ignore'):
            P = np.where(X > 0, XY / X, np.nan)
            R = np.where(Y > 0, XY / Y, np.nan)
            F = np.where(X + Y > 0, 2 * XY / (X + Y), np.nan)
        res[label] = {m: (np.nanpercentile(v, 2.5), np.nanpercentile(v, 97.5))
                      for m, v in [('P', P), ('R', R), ('F1', F)]}
    return res


def ci_table(point_rows, ci, order=('P', 'R', 'F1')):
    """Format 'point [lo, hi]' cells from point estimates and bootstrap CIs."""
    rows = []
    for label, pt in point_rows:
        row = {'Comparison': label}
        for m in order:
            lo, hi = ci[label][m]
            row[m] = f"{pt[m]*100:.1f} [{lo*100:.1f}, {hi*100:.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index('Comparison')

In [4]:
# ── Joint-present rule ────────────────────────────────────────────────────────
# An LLM-extracted issue enters the downstream metrics only if every annotator
# who saw it marked it as present in the judgment. For the 20 shared judgments
# this requires BOTH annotators to have marked it present; for the 30
# single-annotator judgments it reduces to that annotator's own label.
shared_concat_for_joint = pd.concat([
    df_a[df_a['Sentenza'].isin(common_sentenze)],
    df_p[df_p['Sentenza'].isin(common_sentenze)],
], ignore_index=True)
shared_concat_for_joint[COL_PRESENT] = shared_concat_for_joint[COL_PRESENT].astype(bool)

joint_present_lookup = (
    shared_concat_for_joint
    .groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT]
    .all()
)


def with_joint_present(df, common, lookup):
    """Add an `is_present_joint` column to `df`.

    For rows in shared judgments: AND of the two annotators' presence labels
    (an issue is kept only if *both* annotators marked it present).
    For rows in single-annotator judgments: that annotator's own label.
    """
    out = df.copy()
    out['is_present_joint'] = out[COL_PRESENT].astype(bool)
    is_shared = out['Sentenza'].isin(common)
    if is_shared.any():
        keys = pd.MultiIndex.from_arrays([
            out.loc[is_shared, 'Sentenza'].values,
            out.loc[is_shared, 'Questioni estratte'].values,
        ])
        out.loc[is_shared, 'is_present_joint'] = lookup.reindex(keys).values
    return out


df_a = with_joint_present(df_a, common_sentenze, joint_present_lookup)
df_p = with_joint_present(df_p, common_sentenze, joint_present_lookup)
print("is_present_joint flag added.")

is_present_joint flag added.


In [5]:
# ── Combined per-issue frame for the full corpus (N=50) ───────────────────────
def build_combined_df(df_a, df_p, common):
    """Build the per-issue frame used for the Overall (N=50) scores.

    - 30 unique-annotator judgments (15 + 15): rows kept as-is.
    - 20 shared judgments: numeric columns averaged per
      (Sentenza, Questione) across the two annotators.
      `is_present_joint` is identical across the two rows of a shared issue
      (it is the joint AND), so its mean equals that flag.
    """
    df_unique = pd.concat([
        df_a[~df_a['Sentenza'].isin(common)],
        df_p[~df_p['Sentenza'].isin(common)],
    ], ignore_index=True)

    df_shared_concat = pd.concat([
        df_a[df_a['Sentenza'].isin(common)],
        df_p[df_p['Sentenza'].isin(common)],
    ], ignore_index=True)
    df_shared_concat[COL_PRESENT]        = df_shared_concat[COL_PRESENT].astype(int)
    df_shared_concat['is_present_joint'] = df_shared_concat['is_present_joint'].astype(int)

    numeric_cols = df_shared_concat.select_dtypes(include='number').columns.tolist()

    df_shared_avg = (
        df_shared_concat
        .groupby(['Sentenza', 'Questioni estratte'], as_index=False, sort=False)
        .agg({c: 'mean' for c in numeric_cols})
    )

    # COL_NOT_EXTRACTED is a judgment-level field. The per-(Sent, Quest)
    # mean above is unsafe because A1 and A2 sometimes record it on
    # different Questione rows, so a later .max() per judgment would pick
    # the larger of the two instead of their average. Recompute at the
    # judgment level and place it on a single row per judgment.
    a_ne = df_a.groupby('Sentenza')[COL_NOT_EXTRACTED].max().fillna(0)
    p_ne = df_p.groupby('Sentenza')[COL_NOT_EXTRACTED].max().fillna(0)
    shared_ne_avg = (a_ne.loc[list(common)] + p_ne.loc[list(common)]) / 2

    df_shared_avg[COL_NOT_EXTRACTED] = pd.NA
    first_idx = df_shared_avg.drop_duplicates(subset='Sentenza', keep='first').index
    df_shared_avg.loc[first_idx, COL_NOT_EXTRACTED] = (
        df_shared_avg.loc[first_idx, 'Sentenza'].map(shared_ne_avg).values
    )

    return pd.concat([df_unique, df_shared_avg], ignore_index=True)


df_combined = build_combined_df(df_a, df_p, common_sentenze)
assert df_combined['Sentenza'].nunique() == 50, \
    f"Expected 50 judgments in df_combined, got {df_combined['Sentenza'].nunique()}"
print(f"Combined frame: {len(df_combined)} issues over "
      f"{df_combined['Sentenza'].nunique()} judgments")

Combined frame: 78 issues over 50 judgments


## Average number of legal references per present issue — Table `tab:cit_counts`

Expert counts come from each annotator's own citation list; the LLM count is the number of
extracted (post-filter) references. The "LLM, full set" row uses the combined N=50 frame,
in which shared issues have their counts averaged across the two annotators.

In [6]:
rows = []
for label, d in [('A1', df_a), ('A2', df_p)]:
    pres = d[d['is_present_joint'] == True]
    sh = pres[pres['Sentenza'].isin(common_sentenze)]
    rows.append({'': label,
                 'Shared (N=20)': f'{sh[COL_CIT_ESPERTO].mean():.2f} ({int(sh[COL_CIT_ESPERTO].sum())})',
                 'Full set':      f'{pres[COL_CIT_ESPERTO].mean():.2f} ({int(pres[COL_CIT_ESPERTO].sum())})  N=35'})

pres_a = df_a[df_a['is_present_joint'] == True]
sh_llm = pres_a[pres_a['Sentenza'].isin(common_sentenze)]
pres_comb = df_combined[df_combined['is_present_joint'] == True]
rows.append({'': 'LLM',
             'Shared (N=20)': f'{sh_llm[COL_CIT_ESTRATTE].mean():.2f} ({int(sh_llm[COL_CIT_ESTRATTE].sum())})',
             'Full set':      f'{pres_comb[COL_CIT_ESTRATTE].mean():.2f} ({int(pres_comb[COL_CIT_ESTRATTE].sum())})  N=50'})

print('Average number of legal references per present issue (totals in parentheses)')
print(pd.DataFrame(rows).set_index('').to_string())
print(f"\nPresent issues: shared subset {len(sh_llm)}, full corpus {len(pres_comb)}")

Average number of legal references per present issue (totals in parentheses)
    Shared (N=20)          Full set
                                   
A1      2.39 (67)  2.90 (148)  N=35
A2      2.46 (69)  2.82 (144)  N=35
LLM     2.57 (72)  3.08 (228)  N=50

Present issues: shared subset 28, full corpus 74


## Pairwise agreement on citation extraction (N=20, present issues) — Tables `tab:cit_pairwise`, `tab:cit_pairwise_macro`

For each present issue, the annotators and the LLM each produce a set of legal citations.
`shared_citations.csv` records |A1 ∩ A2| per issue (the issue is identified by the last six
characters of its `<text>` field).

For the union/intersection analysis below we also need |LLM ∩ A1 ∩ A2|, which cannot always
be derived from the recorded counts: it is bounded by inclusion–exclusion and we use the
midpoint of the (tight) bounds. For 3 of 28 issues the bounds are not degenerate; the
maximum error is ±1 citation per affected issue (as stated in the paper's footnote).

In [7]:
df_shared_cits = pd.read_csv(f'{DATA}/shared_citations.csv')

cit_records = []
for _, sc_row in df_shared_cits.iterrows():
    sent = sc_row['filename']
    end  = sc_row['issue_end'].strip()

    a_match = df_a[(df_a['Sentenza'] == sent) & (df_a['Questioni estratte'].str.rstrip().str.endswith(end))]
    p_match = df_p[(df_p['Sentenza'] == sent) & (df_p['Questioni estratte'].str.rstrip().str.endswith(end))]
    if len(a_match) == 0 or len(p_match) == 0:
        print(f"WARNING: no match for {sent} [{end}]")
        continue
    a_row = a_match.iloc[0]; p_row = p_match.iloc[0]

    # Skip issues not present in the judgment
    if not a_row[COL_PRESENT] or not p_row[COL_PRESENT]:
        continue

    n_a   = int(a_row[COL_CIT_ESPERTO])    # |A1's citations|
    n_p   = int(p_row[COL_CIT_ESPERTO])    # |A2's citations|
    n_ap  = int(sc_row['shared'])          # |A1 ∩ A2|
    n_llm = int(a_row[COL_CIT_ESTRATTE])   # |LLM citations| (same in both CSVs)
    n_la  = int(a_row[COL_CIT_IN_ESP])     # |LLM ∩ A1|
    n_lp  = int(p_row[COL_CIT_IN_ESP])     # |LLM ∩ A2|

    # |LLM ∩ A1 ∩ A2|: bounded by inclusion-exclusion; use midpoint
    lb = max(0, n_la + n_lp - n_llm)
    ub = min(n_la, n_lp, n_ap)
    n_lap = (lb + ub) / 2

    cit_records.append({
        'Sentenza': sent, 'issue_end': end,
        'n_A': n_a, 'n_P': n_p, 'n_AP': n_ap,
        'n_LLM': n_llm, 'n_LA': n_la, 'n_LP': n_lp,
        'n_LAP': n_lap, 'n_LAP_exact': (lb == ub),
    })

df_cit = pd.DataFrame(cit_records)
n_exact = df_cit['n_LAP_exact'].sum()
print(f"Present issues in shared judgments: {len(df_cit)}")
print(f"|LLM ∩ A1 ∩ A2| exactly determined: {n_exact}/{len(df_cit)}")

Present issues in shared judgments: 28
|LLM ∩ A1 ∩ A2| exactly determined: 26/28


In [8]:
cit_pairs = [
    ('A1 | A2',  'n_A',   'n_P', 'n_AP'),
    ('A2 | A1',  'n_P',   'n_A', 'n_AP'),
    ('LLM | A1', 'n_LLM', 'n_A', 'n_LA'),
    ('LLM | A2', 'n_LLM', 'n_P', 'n_LP'),
]

rows_mi, rows_ma = [], []
for label, cx, cy, cxy in cit_pairs:
    p, r, f = micro_prf1_cols(df_cit, cx, cy, cxy)
    rows_mi.append({'Comparison': label, 'Precision': p, 'Recall': r, 'F1': f})
    p, r, f = macro_prf1_cols(df_cit, cx, cy, cxy)
    rows_ma.append({'Comparison': label, 'Precision': p, 'Recall': r, 'F1': f})

print("POOLED (micro) — Table tab:cit_pairwise:")
print(pd.DataFrame(rows_mi).set_index('Comparison').map(fmt_pct).to_string())
print()
print("PER-ISSUE (macro) — Table tab:cit_pairwise_macro:")
print(pd.DataFrame(rows_ma).set_index('Comparison').map(fmt_pct).to_string())

POOLED (micro) — Table tab:cit_pairwise:
           Precision Recall     F1
Comparison                        
A1 | A2        98.5%  95.7%  97.1%
A2 | A1        95.7%  98.5%  97.1%
LLM | A1       72.2%  77.6%  74.8%
LLM | A2       72.2%  75.4%  73.8%

PER-ISSUE (macro) — Table tab:cit_pairwise_macro:
           Precision Recall     F1
Comparison                        
A1 | A2        99.5%  98.3%  98.9%
A2 | A1        98.3%  99.5%  98.9%
LLM | A1       72.4%  80.1%  76.1%
LLM | A2       72.4%  78.8%  75.5%


### Bootstrap confidence intervals (pooled scores, N=20)

95% percentile intervals from a cluster bootstrap at the judgment level: per-issue
citation counts are first aggregated per judgment, and the judgments are resampled with
replacement (B=10,000). These are the bracketed intervals of the paper's Table
`tab:cit_pairwise`.

In [9]:
cit_count_cols = ['n_A', 'n_P', 'n_AP', 'n_LLM', 'n_LA', 'n_LP']
df_cit_j = df_cit.groupby('Sentenza')[cit_count_cols].sum().reset_index()
print(f"Judgments with at least one present issue (shared set): {len(df_cit_j)}")

ci = bootstrap_micro_prf1(df_cit_j, cit_pairs)
points = [(label, dict(zip(['P', 'R', 'F1'], micro_prf1_cols(df_cit_j, cx, cy, cxy))))
          for label, cx, cy, cxy in cit_pairs]
print("Pooled citation-extraction scores with 95% bootstrap CIs (N=20):")
print(ci_table(points, ci).to_string())

Judgments with at least one present issue (shared set): 20
Pooled citation-extraction scores with 95% bootstrap CIs (N=20):
                             P                   R                  F1
Comparison                                                            
A1 | A2     98.5 [95.7, 100.0]  95.7 [89.9, 100.0]  97.1 [94.1, 100.0]
A2 | A1     95.7 [89.9, 100.0]  98.5 [95.7, 100.0]  97.1 [94.1, 100.0]
LLM | A1     72.2 [53.8, 90.4]   77.6 [68.7, 88.9]   74.8 [62.8, 85.3]
LLM | A2     72.2 [53.8, 90.4]   75.4 [68.3, 85.2]   73.8 [62.4, 83.3]


## LLM citation extraction vs each annotator (N=35) — Tables `tab:cit_n35`, `tab:cit_n35_macro`

Only issues marked as present by the relevant annotator are included.

In [10]:
def cit_counts_per_issue(df_ann):
    sub = df_ann[df_ann[COL_PRESENT] == True].copy()
    return pd.DataFrame({
        'Sentenza': sub['Sentenza'].values,
        'n_llm':    sub[COL_CIT_ESTRATTE].values,
        'n_exp':    sub[COL_CIT_ESPERTO].values,
        'n_match':  sub[COL_CIT_IN_ESP].values,
    })

def llm_cit_vs_annotator(df_r):
    p_mi, r_mi, f_mi = prf1(df_r['n_llm'].sum(), df_r['n_exp'].sum(), df_r['n_match'].sum())
    p_ma, r_ma, f_ma = macro_prf1_cols(df_r, 'n_llm', 'n_exp', 'n_match')
    return {
        'Precision (pooled)': p_mi, 'Recall (pooled)': r_mi, 'F1 (pooled)': f_mi,
        'Precision (per-issue)': p_ma, 'Recall (per-issue)': r_ma, 'F1 (per-issue)': f_ma,
    }

df_cr_a, df_cr_p = cit_counts_per_issue(df_a), cit_counts_per_issue(df_p)
table_cit_n35 = pd.DataFrame({
    'LLM | A1 (N=35)': llm_cit_vs_annotator(df_cr_a),
    'LLM | A2 (N=35)': llm_cit_vs_annotator(df_cr_p),
})
print("LLM citation extraction vs each annotator's full set — Tables tab:cit_n35 (pooled) and tab:cit_n35_macro:")
print(table_cit_n35.map(fmt_pct).to_string())

LLM citation extraction vs each annotator's full set — Tables tab:cit_n35 (pooled) and tab:cit_n35_macro:
                      LLM | A1 (N=35) LLM | A2 (N=35)
Precision (pooled)              72.7%           73.9%
Recall (pooled)                 70.3%           80.6%
F1 (pooled)                     71.5%           77.1%
Precision (per-issue)           74.9%           75.9%
Recall (per-issue)              85.6%           83.8%
F1 (per-issue)                  79.9%           79.6%


In [11]:
# 95% bootstrap CIs for the pooled N=35 scores (Table tab:cit_n35, brackets):
# per-issue counts aggregated per judgment, judgments resampled (B=10,000)
for tag, df_r in [('LLM | A1 (N=35)', df_cr_a), ('LLM | A2 (N=35)', df_cr_p)]:
    df_rj = df_r.groupby('Sentenza')[['n_llm', 'n_exp', 'n_match']].sum().reset_index()
    ci = bootstrap_micro_prf1(df_rj, [(tag, 'n_llm', 'n_exp', 'n_match')])
    pt = dict(zip(['P', 'R', 'F1'], prf1(df_r['n_llm'].sum(), df_r['n_exp'].sum(), df_r['n_match'].sum())))
    print(ci_table([(tag, pt)], ci).to_string())

                                 P                  R                 F1
Comparison                                                              
LLM | A1 (N=35)  72.7 [60.5, 85.8]  70.3 [60.9, 84.2]  71.5 [62.8, 80.7]
                                 P                  R                 F1
Comparison                                                              
LLM | A2 (N=35)  73.9 [60.1, 86.8]  80.6 [73.4, 88.1]  77.1 [66.9, 85.3]


## Union / intersection ground truth (N=50) — Appendix Table `tab:cit_union_inter`

For the 20 shared judgments, reference citations are defined as:
**union** = cited by *either* annotator (larger reference, harder recall);
**intersection** = cited by *both* annotators (stricter reference). For the 30 unique
judgments there is only one annotator. |LLM ∩ A1 ∩ A2| uses the midpoint approximation
described above.

In [12]:
only_a = sentenze_a - common_sentenze
only_p = sentenze_p - common_sentenze
cit_50_records = []

# Unique judgments: one annotator, present issues only
for sent in only_a:
    for _, r in df_a[(df_a['Sentenza'] == sent) & (df_a[COL_PRESENT] == True)].iterrows():
        cit_50_records.append({
            'n_llm': r[COL_CIT_ESTRATTE],
            'n_ref_union': r[COL_CIT_ESPERTO], 'n_ref_inter': r[COL_CIT_ESPERTO],
            'n_match_union': r[COL_CIT_IN_ESP], 'n_match_inter': r[COL_CIT_IN_ESP],
        })

for sent in only_p:
    for _, r in df_p[(df_p['Sentenza'] == sent) & (df_p[COL_PRESENT] == True)].iterrows():
        cit_50_records.append({
            'n_llm': r[COL_CIT_ESTRATTE],
            'n_ref_union': r[COL_CIT_ESPERTO], 'n_ref_inter': r[COL_CIT_ESPERTO],
            'n_match_union': r[COL_CIT_IN_ESP], 'n_match_inter': r[COL_CIT_IN_ESP],
        })

# Shared judgments: use df_cit
for _, row in df_cit.iterrows():
    n_union_ref  = row['n_A'] + row['n_P'] - row['n_AP']
    n_inter_ref  = row['n_AP']
    n_match_union = row['n_LA'] + row['n_LP'] - row['n_LAP']  # |LLM∩(A1∪A2)|
    n_match_inter = row['n_LAP']                              # |LLM∩(A1∩A2)|

    cit_50_records.append({
        'n_llm': row['n_LLM'],
        'n_ref_union': n_union_ref,   'n_ref_inter': n_inter_ref,
        'n_match_union': n_match_union, 'n_match_inter': n_match_inter,
    })

df_cit_50 = pd.DataFrame(cit_50_records)
print(f"Total present issues across N=50 judgments: {len(df_cit_50)}")

table_cit_ui = pd.DataFrame({
    'Union (N=50)': {
        'Precision (pooled)':    micro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_union', 'n_match_union')[0],
        'Recall (pooled)':       micro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_union', 'n_match_union')[1],
        'Precision (per-issue)': macro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_union', 'n_match_union')[0],
        'Recall (per-issue)':    macro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_union', 'n_match_union')[1],
    },
    'Intersection (N=50)': {
        'Precision (pooled)':    micro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[0],
        'Recall (pooled)':       micro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[1],
        'Precision (per-issue)': macro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[0],
        'Recall (per-issue)':    macro_prf1_cols(df_cit_50, 'n_llm', 'n_ref_inter', 'n_match_inter')[1],
    },
})

print("\nLLM citation extraction with union/intersection ground truth — Table tab:cit_union_inter:")
print(table_cit_ui.map(fmt_pct).to_string())

Total present issues across N=50 judgments: 74

LLM citation extraction with union/intersection ground truth — Table tab:cit_union_inter:
                      Union (N=50) Intersection (N=50)
Precision (pooled)           74.3%               73.0%
Recall (pooled)              75.0%               75.0%
Precision (per-issue)        77.0%               75.9%
Recall (per-issue)           87.1%               85.9%


## Citation-reason faithfulness — Table `tab:cit_reason`

Each annotator scored every LLM-extracted `<citation_reason>` on a binary scale (1 if the
stated reason corresponds to the use the judge makes of the citation, 0 otherwise) and
recorded the per-issue **sum**. The faithfulness rate pools reasons across issues:
(sum of scores) / (number of extracted reasons). The LLM full-corpus rate uses the
combined N=50 frame. We also report the inter-annotator exact agreement on the per-issue
count of faithful reasons (in-text number).

In [13]:
rows = []
for label, d in [('A1, shared', df_a), ('A2, shared', df_p)]:
    sh = d[(d['Sentenza'].isin(common_sentenze)) & (d['is_present_joint'] == True)]
    n_reasons = sh[COL_MOTIVI_ESTRATTI].fillna(0).sum()
    rate = sh[COL_SCORE_MOTIVI].fillna(0).sum() / n_reasons
    rows.append({'': label, 'N reasons': int(n_reasons), 'Faithfulness rate': f'{rate:.1%}'})

pres_comb = df_combined[df_combined['is_present_joint'] == True]
n_reasons_llm = pres_comb[COL_MOTIVI_ESTRATTI].fillna(0).sum()
rate_llm = pres_comb[COL_SCORE_MOTIVI].fillna(0).sum() / n_reasons_llm
rows.append({'': 'LLM, full corpus (N=50)', 'N reasons': int(round(n_reasons_llm)),
             'Faithfulness rate': f'{rate_llm:.1%}'})

print('Mean faithfulness rate (pooled) — Table tab:cit_reason:')
print(pd.DataFrame(rows).set_index('').to_string())

Mean faithfulness rate (pooled) — Table tab:cit_reason:
                         N reasons Faithfulness rate
                                                    
A1, shared                      49             81.6%
A2, shared                      49             75.5%
LLM, full corpus (N=50)        143             80.8%


In [14]:
# Inter-annotator exact agreement on the per-issue count of faithful reasons
# (shared judgments, issues marked present by both annotators; the number of
# extracted reasons per issue is identical in the two annotators' files).
df_sh = pd.concat([df_a.assign(annotator='A1'), df_p.assign(annotator='A2')], ignore_index=True)
df_sh = df_sh[df_sh['Sentenza'].isin(common_sentenze)]
df_valid = (df_sh.groupby(['Sentenza', 'Questioni estratte'])
                 .filter(lambda g: g[COL_PRESENT].all()))

piv = (df_valid.dropna(subset=[COL_SCORE_MOTIVI])
              .pivot_table(index=['Sentenza', 'Questioni estratte'], columns='annotator',
                           values=COL_SCORE_MOTIVI, aggfunc='first')
              .dropna())

exact = (piv['A1'] == piv['A2']).mean()
print(f"Issues compared: {len(piv)}")
print(f"Exact agreement on the count of faithful reasons: {exact:.1%}")

Issues compared: 28
Exact agreement on the count of faithful reasons: 96.4%
